# Exploratory Data Analysis (EDA) & Visualisasi Data
## Kualitas Udara di Indonesia (2022 - 2026)
**Projek Praktikum Pemrograman - Pipeline ETL & Visualisasi**

Notebook ini menyajikan analisis data eksploratif (EDA) dan visualisasi grafik terhadap dataset kualitas udara Indonesia hasil pembersihan pipeline ETL.


### 1. Inisialisasi Library dan Konfigurasi Visualisasi

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Atur konfigurasi tampilan grafik
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.autolayout'] = True
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print('Semua library berhasil dimuat.')


### 2. Memuat Dataset Langsung dari Data Warehouse (SQL Query)
Data diambil langsung dari tabel relasional `air_quality_daily` dan `air_quality_province_summary` di dalam database `air_quality_warehouse.db` menggunakan kueri SQL.


In [ ]:
import sqlite3
# Koneksi langsung ke Data Warehouse relasional lokal
db_path = "../data/processed/air_quality_warehouse.db"
conn = sqlite3.connect(db_path)
df_daily = pd.read_sql_query("SELECT * FROM air_quality_daily", conn)
df_summary = pd.read_sql_query("SELECT * FROM air_quality_province_summary", conn)
conn.close()

print(f"Terhubung ke Data Warehouse: {db_path}")
print(f"Total Baris Data Harian (Tabel air_quality_daily): {len(df_daily):,}")
print(f"Total Baris Ringkasan Provinsi (Tabel air_quality_province_summary): {len(df_summary):,}")
df_daily.head()


### 3. Pemeriksaan Struktur dan Statistik Deskriptif

In [ ]:
# Memeriksa tipe data dan kelengkapan kolom
df_daily.info()


In [ ]:
# Ringkasan statistik metrik numerik utama
df_daily[['pm2_5_mean', 'pm10_mean', 'us_aqi_mean', 'hours']].describe().round(2)


### 4. Visualisasi 1: Proporsi Status Kualitas Udara Nasional (Donut Chart)
Grafik ini memperlihatkan bagaimana distribusi status udara Indonesia menurut standar US AQI EPA.


In [ ]:
cat_order = ['Baik', 'Sedang', 'Tidak Sehat Kelompok Sensitif', 'Tidak Sehat', 'Sangat Tidak Sehat', 'Berbahaya']
colors = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad', '#5d4037']

cat_counts = df_daily['aqi_category'].value_counts().reindex(cat_order).dropna()

plt.figure(figsize=(7, 7))
plt.pie(
    cat_counts, 
    labels=cat_counts.index, 
    autopct='%1.1f%%',
    pctdistance=0.8,
    colors=colors,
    startangle=140,
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2)
)
plt.title('Proporsi Kategori Kualitas Udara di Indonesia (2022 - 2026)', fontsize=13, weight='bold', pad=20)
plt.show()


### 5. Visualisasi 2: 5 Provinsi Terbersih vs 5 Provinsi Paling Berpolusi
Membandingkan disparitas kualitas udara antar provinsi berdasarkan rata-rata indeks US AQI.


In [ ]:
prov_avg = df_daily.groupby('province')['us_aqi_mean'].mean().reset_index()
worst_5 = prov_avg.sort_values(by='us_aqi_mean', ascending=False).head(5)
best_5 = prov_avg.sort_values(by='us_aqi_mean', ascending=True).head(5)
comparison = pd.concat([worst_5, best_5]).sort_values(by='us_aqi_mean', ascending=True)

plt.figure(figsize=(9, 5.5))
bar_colors = ['#27ae60' if val < 50 else ('#e67e22' if val < 150 else '#e74c3c') for val in comparison['us_aqi_mean']]
bars = plt.barh(comparison['province'], comparison['us_aqi_mean'], color=bar_colors, edgecolor='gray', alpha=0.85)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 2, bar.get_y() + bar.get_height()/2, f'{w:.1f}', va='center', fontsize=9, weight='bold')

plt.axvline(x=50, color='#27ae60', linestyle='--', label='Batas Udara Baik (AQI <= 50)')
plt.axvline(x=100, color='#e67e22', linestyle='--', label='Batas Mulai Tidak Sehat (AQI > 100)')
plt.title('5 Provinsi Paling Berpolusi vs 5 Provinsi Terbersih di Indonesia', fontsize=12, weight='bold', pad=15)
plt.xlabel('Rata-rata US AQI')
plt.xlim(0, 180)
plt.legend(loc='lower right')
plt.show()


### 6. Visualisasi 3: Ketimpangan Pelampauan Batas Aman Standar WHO per Gugus Pulau
Standar WHO menetapkan batas konsentrasi PM2.5 harian maksimal adalah 15 µg/m³. Grafik ini memperlihatkan persentase hari yang gagal memenuhi standar tersebut.


In [ ]:
island_who = (df_daily.groupby('island')['exceeds_who_pm25'].mean() * 100).sort_values(ascending=False)

plt.figure(figsize=(8.5, 5))
palette = sns.color_palette('Reds_r', n_colors=len(island_who))
bars = plt.bar(island_who.index, island_who.values, color=palette, edgecolor='black', linewidth=0.5)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 1.5, f'{h:.1f}%', ha='center', fontsize=9, weight='bold')

plt.title('Persentase Hari Melebihi Batas Standar WHO (PM2.5 > 15 µg/m³)', fontsize=12, weight='bold', pad=15)
plt.ylabel('% Hari Melampaui Batas WHO')
plt.xlabel('Gugus Kepulauan')
plt.ylim(0, 105)
plt.xticks(rotation=20, ha='right')
plt.show()


### 7. Visualisasi 4: Tren Polusi Udara Tahunan (2022 - 2026)

In [ ]:
yearly = df_daily.groupby('year')[['pm2_5_mean', 'us_aqi_mean']].mean().reset_index()

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(yearly['year'], yearly['us_aqi_mean'], color='#d35400', marker='o', linewidth=2.5, label='US AQI (Rata-rata)')
ax1.set_xlabel('Tahun')
ax1.set_ylabel('Rata-rata US AQI', color='#d35400', weight='bold')
ax1.tick_params(axis='y', labelcolor='#d35400')
ax1.set_xticks(yearly['year'])

ax2 = ax1.twinx()
ax2.plot(yearly['year'], yearly['pm2_5_mean'], color='#2980b9', marker='s', linewidth=2.5, linestyle='--', label='PM2.5 (µg/m³)')
ax2.set_ylabel('Konsentrasi PM2.5 (µg/m³)', color='#2980b9', weight='bold')
ax2.tick_params(axis='y', labelcolor='#2980b9')
ax2.grid(False)

plt.title('Tren Kenaikan Polusi Udara Tahunan di Indonesia (2022 - 2026)', fontsize=12, weight='bold', pad=15)
plt.show()


### 8. Visualisasi 5: Pola Musiman (Pengaruh Musim Kemarau vs Hujan)

In [ ]:
df_daily['date'] = pd.to_datetime(df_daily['date'])
df_daily['month'] = df_daily['date'].dt.month

monthly_java = df_daily[df_daily['island'] == 'Java'].groupby('month')['us_aqi_mean'].mean()
monthly_non_java = df_daily[df_daily['island'] != 'Java'].groupby('month')['us_aqi_mean'].mean()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des']

plt.figure(figsize=(9, 5))
plt.plot(month_names, monthly_java.values, marker='o', color='#c0392b', linewidth=2, label='Pulau Jawa')
plt.plot(month_names, monthly_non_java.values, marker='s', color='#2980b9', linestyle='--', linewidth=2, label='Luar Jawa')
plt.axvspan('Jun', 'Okt', color='#f39c12', alpha=0.15, label='Puncak Musim Kemarau')

plt.title('Pola Musiman Polusi Udara (Perbandingan Pulau Jawa vs Luar Jawa)', fontsize=12, weight='bold', pad=15)
plt.xlabel('Bulan')
plt.ylabel('Rata-rata US AQI')
plt.legend(loc='lower left')
plt.show()


### 9. Kesimpulan Utama EDA
1. **Ketimpangan Ekstrem:** Pulau Jawa mengalami darurat polusi (hampir 89% hari melanggar batas WHO), dipimpin oleh DKI Jakarta dan Jawa Barat.
2. **Wilayah Paling Sehat:** Indonesia Timur (Papua dan Maluku) memiliki kualitas udara paling bersih di Indonesia (rata-rata AQI < 40).
3. **Pengaruh Musim:** Polusi udara memuncak secara signifikan pada musim kemarau (Juli–Oktober) akibat minimnya presipitasi hujan dan peningkatan akumulasi partikulat di atmosfer.
4. **Tren Meningkat:** Terdapat tren peningkatan polusi secara bertahap dari tahun 2022 hingga 2026.
